## Method 4

In [5]:
# If needed:
# pip install transformers datasets torch accelerate
from dataclasses import dataclass
from datetime import date
from typing import List

import pandas as pd
import numpy as np
import torch

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.metrics import accuracy_score, precision_recall_fscore_support

##### Define the Critique dataclass

In [6]:
@dataclass
class Critique:
    critique_id: str
    painting_id: str
    critic_name: str
    title: str
    text: str
    published_at: date
    label: int

##### Define the critiques

In [7]:

critiques: List[Critique] = [

    Critique(
        critique_id="E1",
        painting_id="P1",
        critic_name="Expert A",
        title="Composition and Structure",
        text="The painting demonstrates strong compositional balance, spatial depth, controlled movement, and a carefully organized visual hierarchy.",
        published_at=date(2021, 5, 12),
        label=1
    ),

    Critique(
        critique_id="E2",
        painting_id="P1",
        critic_name="Expert B",
        title="Color and Light",
        text="The artist uses a restrained color palette, dramatic illumination, and subtle tonal contrast to create emotional tension.",
        published_at=date(2022, 3, 18),
        label=1
    ),

    Critique(
        critique_id="E3",
        painting_id="P1",
        critic_name="Expert C",
        title="Psychological Reading",
        text="The figures and shadows suggest inner conflict, emotional ambiguity, and a psychologically charged interpretation of the scene.",
        published_at=date(2023, 1, 9),
        label=1
    ),

    Critique(
        critique_id="E4",
        painting_id="P1",
        critic_name="Expert D",
        title="Historical Context",
        text="The work reflects the visual language of its historical period, connecting religious symbolism, patronage, and cultural context.",
        published_at=date(2020, 10, 5),
        label=1
    ),

    Critique(
        critique_id="E5",
        painting_id="P1",
        critic_name="Expert E",
        title="Symbol and Story",
        text="The symbolic gestures, placement of figures, and narrative details invite an iconographic reading of the painting.",
        published_at=date(2021, 8, 22),
        label=1
    ),

    Critique(
        critique_id="S1",
        painting_id="P1",
        critic_name="Student 1",
        title="Drama Through Light and Placement",
        text="The painting uses light and shadow to create a serious mood. The placement of the figures makes the scene feel dramatic and meaningful.",
        published_at=date(2024, 2, 12),
        label=0
    ),

    Critique(
        critique_id="S2",
        painting_id="P1",
        critic_name="Student 2",
        title="Emotional Response",
        text="The dark tones and facial expressions create a feeling of sadness and tension. The shadows make the scene feel emotionally powerful and mysterious.",
        published_at=date(2024, 2, 13),
        label=0
    ),

    Critique(
        critique_id="S3",
        painting_id="P1",
        critic_name="Student 3",
        title="Formal Analysis Attempt",
        text="The composition guides the viewer through the scene using contrast, spatial depth, and careful placement of figures. The lighting reinforces the emotional intensity of the painting.",
        published_at=date(2024, 2, 14),
        label=0
    ),

    Critique(
        critique_id="S4",
        painting_id="P1",
        critic_name="Student 4",
        title="General Impression",
        text="I think the artist wanted to create a dramatic atmosphere. The painting is beautiful and interesting, although some parts are difficult to understand.",
        published_at=date(2024, 2, 15),
        label=0
    ),

    Critique(
        critique_id="S5",
        painting_id="P1",
        critic_name="Student 5",
        title="Symbolic Interpretation",
        text="The arrangement of the figures and the use of shadow may symbolize conflict or isolation. The visual structure suggests a deeper psychological meaning beneath the surface narrative.",
        published_at=date(2024, 2, 16),
        label=0
    ),
]

df = pd.DataFrame([c.__dict__ for c in critiques])

df[["critique_id", "critic_name", "title", "label"]]

,critique_id,critic_name,title,label
0,E1,Expert A,Composition and Structure,1
1,E2,Expert B,Color and Light,1
2,E3,Expert C,Psychological Reading,1
3,E4,Expert D,Historical Context,1
4,E5,Expert E,Symbol and Story,1
5,S1,Student 1,Drama Through Light and Placement,0
6,S2,Student 2,Emotional Response,0
7,S3,Student 3,Formal Analysis Attempt,0
8,S4,Student 4,General Impression,0
9,S5,Student 5,Symbolic Interpretation,0


##### Convert DataFrame to a HuggingFace Dataset

In [8]:
dataset = Dataset.from_pandas(
    df[["text", "label"]]
)


##### Load Tokenizer

In [9]:
#Load Tokenizer
model_checkpoint = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(
    model_checkpoint
)

##### Tokenize Text

In [ ]:
#Tokenize Text
def tokenize_function(example):

    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

tokenized_dataset = dataset.map(tokenize_function)

##### Split train/test sets

In [7]:
split_dataset = tokenized_dataset.train_test_split(
    test_size=0.3,
    seed=42
)

train_dataset = split_dataset["train"]
test_dataset = split_dataset["test"]

##### Load transformer classifier

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=2
)

##### Define Evaluation Metrics

In [9]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="binary"
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

##### Training Configuration

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",

    # evaluation_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,

    num_train_epochs=100,

    weight_decay=0.01,

    logging_dir="./logs",

    logging_steps=5
)

##### Create Trainer

In [11]:
trainer = Trainer(
    model=model,

    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=test_dataset,

    compute_metrics=compute_metrics
)

##### Fine-tune the transformer

In [ ]:
trainer.train()

##### Evaluate Model

In [13]:
evaluation_results = trainer.evaluate()

evaluation_results

{'eval_loss': 0.03979087248444557,
 'eval_accuracy': 1.0,
 'eval_precision': 1.0,
 'eval_recall': 1.0,
 'eval_f1': 1.0,
 'eval_runtime': 0.1185,
 'eval_samples_per_second': 25.311,
 'eval_steps_per_second': 16.874,
 'epoch': 100.0}

##### Predict a New Critique

In [14]:

new_critique = """
The arrangement of the figures and the careful use of shadow
create psychological tension and symbolic ambiguity throughout the composition.
"""

inputs = tokenizer(
    new_critique,
    return_tensors="pt",
    truncation=True,
    padding=True
)

# Move inputs to the same device as the model
device = model.device
inputs = {key: value.to(device) for key, value in inputs.items()}

model.eval()

with torch.no_grad():
    outputs = model(**inputs)

    probabilities = torch.softmax(
        outputs.logits,
        dim=1
    )

predicted_label = torch.argmax(
    probabilities,
    dim=1
).item()

confidence = probabilities[0][predicted_label].item()

print("Predicted label:", predicted_label)
print("Confidence:", round(confidence, 3))

print("Probability novice-like:",
      round(probabilities[0][0].item(), 3))

print("Probability expert-like:",
      round(probabilities[0][1].item(), 3))

print("device:", device)

Predicted label: 0
Confidence: 0.685
Probability novice-like: 0.685
Probability expert-like: 0.315
device: cuda:0
